In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import AdamW
import random
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Bidirectional, Dense, Dropout


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


This loads the dataframe and filters out any duplicated lines and any empty lines if there is any.




In [3]:
#Loading the dataset and then dropping any null values just incase
df = pd.read_csv('/content/drive/MyDrive/Emotion_detection/Emotion-Dataset.csv')  # text and emotion columns
#get the orginal lenght of the comlumns
og_def_len = len(df)
#drop any empty lines, this isnt something that ive noticed but its just incase
df.dropna(inplace=True)
#check to see if theres any duplicates
df = df.drop_duplicates(subset='Text')
#Train/Test overlap: 3152 rows this was the number of leaked data that matched between my test and train data we end up filtering 7,767
filtered_def_len = len(df)

print(f"number of filtered lines: {og_def_len-filtered_def_len}")

number of filtered lines: 7767


The following code makes sure all the emotions are lowercase then uses a set to get the catagories to a dictionary

In [46]:
#make the emotions lowercase
df_emotion_lower = df['Emotion'].str.lower()

#leveraging sets to get 1 of each emotion i noticed there is fear and Fear i decided to make it all lower case to avoid duplicates
Types_of_emotions = set(df_emotion_lower)
print(f"list of emotion catagories:{Types_of_emotions}")
emotion_to_number = {}
#adding a number value to the emotions
for idx in enumerate(Types_of_emotions):
    emotion = idx[1] # getting the emotion
    number = idx[0] # getting the number position
    emotion_to_number[emotion] = number #assigning it to a dictonary



list of emotion catagories:{'anger', 'joy', 'sadness', 'fear', 'surprise'}
0      sadness
1         fear
2        anger
3     surprise
4         fear
5        anger
6         fear
7        anger
8          joy
9         fear
10         joy
11     sadness
12       anger
13         joy
14       anger
15       anger
16         joy
17         joy
18        fear
19     sadness
Name: Emotion, dtype: object


Replacing the emotion column with all lowercase then we are assigning the numeric labels to a new column

In [5]:
#Replacing lowercase emotions in the dataframe
df['Emotion'] = df_emotion_lower

#Creating a numbered column
df['Emotion_Label'] = None

#I am now looping through the dataframe checking the emotion and assigning the label that was created from dictionary might not be the most efficent but its what made sense to me
for index, row in df.iterrows():
    emotion = row['Emotion']  #grab the emotion string from the row
    if emotion in emotion_to_number:
        df.at[index, 'Emotion_Label'] = emotion_to_number[emotion]




Testing a small sample and giving a chance to visualize whats changed in the dataframe

In [6]:
#testing a small sample size of thet text, emotion and label.
print(df[['Text','Emotion', 'Emotion_Label']].head(10))
print(f"Total rows in the dataset: {len(df)}")


                                                Text   Emotion Emotion_Label
0  i could get depressed about feeling isolated b...   sadness             2
1  i am so thankful that though things are a bit ...      fear             3
2  i remember one day years ago when the kids wer...     anger             0
3  i feel so funny he have no topic to chat with ...  surprise             4
4  i didnt take it personally but i could feel so...      fear             3
5  i often feel rushed and comfort sitting on a l...     anger             0
6  i feel as though my mind is restless searching...      fear             3
7  i feel as if they rushed these out to the publ...     anger             0
8  i can pound out something i feel works really ...       joy             1
9  i find god s presence if i feel scared lonely ...      fear             3
Total rows in the dataset: 142233


This is to check the emotions labels and make sure they match up to the dictionary keys

In [7]:

label_map = {emotion: idx for idx, emotion in enumerate(set(df['Emotion']))}
print(label_map)

{'anger': 0, 'joy': 1, 'sadness': 2, 'fear': 3, 'surprise': 4}


Now we are splitting the data i decided to drop the emotion and just keep the number seemed reduant


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    df['Text'], df['Emotion_Label'], test_size=0.3, random_state=24
)


Tokenizing and padding sequences

In [9]:


#Putting the text into tokenized sequences
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)
vocab_size = len(tokenizer.word_index) + 1
max_len = 140  # I increased this so it can see more of the sentence and context
#applying a padding to make sure its all the same lenght even if its less than 140
X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len)
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len)

# y_train and y_test already contain the numeric Emotion_Label values
# But we need to make sure they're NumPy arrays with the right data type
y_train = np.array(y_train, dtype=np.int64)
y_test = np.array(y_test, dtype=np.int64)
num_classes = len(label_map)

Declare model and paramaters and test and train

In [48]:


#Adding learning rate and opotmizer
learning_rate = 0.026
optimizer = AdamW(learning_rate=learning_rate, weight_decay=1e-1)  #testing weight decay i aded a low amount in a hope to get a slight increase and break

#Model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=140),
    GRU(128, return_sequences=True, dropout=0.5),  # Added dropout to prevent overfitting
    GRU(64, dropout=0.5),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')  # I experimented with a different final activation function and softmax worked much better than two relu
])

#compling the model
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

#Model training

history = model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=8192,  # I realized that if it sees more sentences its able to learn the information quicker.
    validation_data=(X_test_pad, y_test),
)

#Generate classification score and evlauating the accuracy
test_loss, test_accuracy = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy:.4f}")


y_pred = model.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred, axis=1)

print("\nF1 scores by emotion:")
print(classification_report(y_test, y_pred_classes))

#Saving model and checkpoint
model.save('emotion_gru_model.h5')


Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 234ms/step - accuracy: 0.2655 - loss: 1.6051 - val_accuracy: 0.3972 - val_loss: 1.4206
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 200ms/step - accuracy: 0.4991 - loss: 1.2644 - val_accuracy: 0.8828 - val_loss: 0.3833
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 198ms/step - accuracy: 0.9080 - loss: 0.3115 - val_accuracy: 0.9578 - val_loss: 0.1332
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 199ms/step - accuracy: 0.9638 - loss: 0.1162 - val_accuracy: 0.9684 - val_loss: 0.0808
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 197ms/step - accuracy: 0.9726 - loss: 0.0743 - val_accuracy: 0.9695 - val_loss: 0.0737
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 200ms/step - accuracy: 0.9748 - loss: 0.0625 - val_accuracy: 0.9709 - val_loss: 0.0668
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 199ms/step - accuracy: 0.9778 - loss: 0.0523 - val_accuracy: 0.9708 - val_loss: 0.0713
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 199ms/step - accuracy: 0.9787 - loss: 0.0491 - val_accuracy: 0.


F1 scores by emotion:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      8883
           1       0.99      0.99      0.99     12165
           2       0.98      0.98      0.98     11979
           3       0.95      0.93      0.94      7309
           4       0.86      0.94      0.90      2334

    accuracy                           0.97     42670
   macro avg       0.95      0.96      0.96     42670
weighted avg       0.97      0.97      0.97     42670



Sanity test to make sure there isnt any overlap between whats in the testing and training sets

In [11]:
overlap = set(X_train).intersection(set(X_test))
print(f"Train/Test overlap: {len(overlap)} rows")
#checking to see why my accuracy is over 90 percent right away and if theres any leaks i believe there might be duplicates


Train/Test overlap: 0 rows


Show 50 random samples to view the predictions and Ground truth

In [27]:
#reverting from numeric to text
number_to_emotion = {v: k for k, v in emotion_to_number.items()}

#Pick 50 random samples from test set
sample_indices = random.sample(range(len(X_test)), 10000)
sample_texts = X_test.iloc[sample_indices].tolist()
sample_true_labels = y_test[sample_indices]
sample_pred_classes = y_pred_classes[sample_indices]  # ← pull from already predicted classes!

# Show results
print("Emotion Predictions on 50 Test Sentences")
for i, text in enumerate(sample_texts):
    pred_emotion = number_to_emotion[sample_pred_classes[i]]
    true_emotion = number_to_emotion[sample_true_labels[i]]
    print(f"{i+1}. \"{text}\"")
    print(f"   → Predicted: {pred_emotion} | Actual: {true_emotion}\n")




Emotion Predictions on 50 Test Sentences
1. "im feeling sad so i can remind myself of how i am talented and good at things and also see things that inspire me all in once place"
   → Predicted: sadness | Actual: sadness

2. "i was able to identify with a lot of the reasons and although i found the beginning bit really triggering it also made me feel as though i wasnt alone and helped me understand myself a bit more"
   → Predicted: sadness | Actual: sadness

3. "i opted to shrug it off but lately i kinda feel how he distances himself from me how he sometimes selectively not hear me when i talk and most of the time he makes me feel unwelcome with his words and gestures"
   → Predicted: sadness | Actual: sadness

4. "i am feeling less lethagic and miserable the fridge is stocked with various animal products and a ton of fruit and veg after this morning s shop and onward we go into the th month since we started trying for no"
   → Predicted: sadness | Actual: sadness

5. "i don t feel lik

High Confidence errors


In [118]:
#current model accuracy
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len)
initial_test_loss, initial_test_acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Model accuracy: {initial_test_acc:.4f}")

#Find errors in general
y_pred = model.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred, axis=1)
misclassified_indices = np.where(y_pred_classes != y_test)[0]
print(f"Found {len(misclassified_indices)} misclassified examples out of {len(y_test)} test examples")

#Sort high confidence errors
confidences = np.max(y_pred[misclassified_indices], axis=1)
high_conf_errors = np.where(confidences > 0.9)[0]
print(f"\nFound {len(high_conf_errors)} high confidence errors (>90% confidence)")

#Display high confidence errors
print("\nHigh confidence errors:")
for i in range(min(10, len(high_conf_errors))):
    idx = misclassified_indices[high_conf_errors[i]]
    text = X_test.iloc[idx]
    true_emotion = number_to_emotion[y_test[idx]]
    pred_emotion = number_to_emotion[y_pred_classes[idx]]
    conf = confidences[high_conf_errors[i]]

    print(f"\nLabeled as: {true_emotion}, Model thinks: {pred_emotion}, Confidence: {conf:.4f}")
    print(f"Text: \"{text}\"")

Model accuracy: 0.9696
1334/1334 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step
Found 1293 misclassified examples out of 42670 test examples

Found 308 high confidence errors (>90% confidence)

High confidence errors:

Labeled as: anger, Model thinks: fear, Confidence: 0.9256
Text: "i havent done much reading like im suppose to be ive been feeling agitated and confused since yesterday i had a really sad dream last night todays early morning"

Labeled as: sadness, Model thinks: anger, Confidence: 0.9953
Text: "the death of my guardian with whom i had stayed when i did my grade six"

Labeled as: anger, Model thinks: fear, Confidence: 0.9446
Text: "i don t want to go home to toronto and feel like a nobody tortured artist loser for two weeks and smoke pot alone in my bedroom and watch degrassi junior high and then weep"

Labeled as: sadness, Model thinks: fear, Confidence: 0.9426
Text: "i no doubt will also be angry at myself for feeling so helpless and not feeling independent and for not being able to 